[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C20_Frontier_Architectures_Course/03_moe/03_moe.ipynb)

# 03 · MoE 与 SwiGLU（从零实现）

目标：手写 SwiGLU、top-k 路由、MoE 前向、负载均衡损失，并统计专家利用率。

路线：SwiGLU → 路由 → MoE 前向 → 均衡损失 → 利用率 → ✏️ 练习 → 🧪 真实稀疏度胶囊。

## 1 · SwiGLU 门控 FFN

`SwiGLU(x) = (Swish(xWg) ⊙ (xWu)) Wd`，`Swish(z)=z·sigmoid(z)`。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def silu(z):
    return z / (1.0 + np.exp(-z))   # Swish/SiLU

def swiglu(x, Wg, Wu, Wd):
    return (silu(x @ Wg) * (x @ Wu)) @ Wd

d_model, d_hidden = 16, 32
Wg = rng.standard_normal((d_model, d_hidden)) / np.sqrt(d_model)
Wu = rng.standard_normal((d_model, d_hidden)) / np.sqrt(d_model)
Wd = rng.standard_normal((d_hidden, d_model)) / np.sqrt(d_hidden)
x = rng.standard_normal(d_model)
print('SwiGLU 输出形状', swiglu(x, Wg, Wu, Wd).shape)
assert swiglu(x, Wg, Wu, Wd).shape == (d_model,)

## 2 · 路由器与 top-k

路由 logits → softmax → 取 top-k → 在 k 个上重新归一化门控。

In [ ]:
def route(X, Wr, k):
    logits = X @ Wr                      # (T, N)
    probs = softmax(logits, axis=-1)
    topk = np.argsort(-probs, axis=1)[:, :k]          # (T, k) 专家下标
    gates = np.take_along_axis(probs, topk, axis=1)   # (T, k)
    gates = gates / gates.sum(axis=1, keepdims=True)  # 重新归一化
    return topk, gates, probs

T, N, k = 12, 6, 2
X = rng.standard_normal((T, d_model))
Wr = rng.standard_normal((d_model, N)) / np.sqrt(d_model)
topk, gates, probs = route(X, Wr, k)
print('每个 token 的 top-2 专家:\n', topk[:4])
print('门控权重每行和应为 1：', np.allclose(gates.sum(1), 1))
assert np.allclose(gates.sum(1), 1)

## 3 · MoE 前向

每个 token 只过它的 top-k 专家，加权求和。

In [ ]:
def make_experts(N, d_model, d_hidden):
    return [ (rng.standard_normal((d_model, d_hidden))/np.sqrt(d_model),
             rng.standard_normal((d_model, d_hidden))/np.sqrt(d_model),
             rng.standard_normal((d_hidden, d_model))/np.sqrt(d_hidden)) for _ in range(N) ]

def moe_forward(X, Wr, experts, k):
    topk, gates, probs = route(X, Wr, k)
    out = np.zeros_like(X)
    for t in range(X.shape[0]):
        for j in range(k):
            e = topk[t, j]
            out[t] += gates[t, j] * swiglu(X[t], *experts[e])
    return out, topk, probs

experts = make_experts(N, d_model, d_hidden)
out, topk, probs = moe_forward(X, Wr, experts, k)
active = k / N
print('MoE 输出形状', out.shape, f'| 每 token 激活 {k}/{N} 专家 = {active:.0%} 算力')
assert out.shape == (T, d_model)

## 4 · 负载均衡损失

`L_aux = α·N·Σ f_i·P_i`，`f_i`=分配占比，`P_i`=平均概率。均匀时最小。

In [ ]:
def load_balance_loss(topk, probs, N, alpha=0.01):
    T, k = topk.shape
    counts = np.bincount(topk.reshape(-1), minlength=N)
    f = counts / (T * k)                 # f_i: 分配占比
    P = probs.mean(axis=0)               # P_i: 平均概率
    return alpha * N * np.sum(f * P), f, P

loss, f, P = load_balance_loss(topk, probs, N)
print('负载均衡损失 =', round(loss, 5))
print('分配占比 f =', np.round(f, 3))
print('均匀基线（理想）≈ α =', 0.01, '（f=P=1/N 时 α·N·N·(1/N)^2 = α）')
assert loss > 0

## 5 · 专家利用率

坍塌的征兆：少数专家吃掉大部分 token。统计利用率分布与变异系数 CV。

In [ ]:
counts = np.bincount(topk.reshape(-1), minlength=N)
for i, ct in enumerate(counts):
    bar = '#' * int(ct)
    print(f'专家 {i}: {ct:2d} tokens {bar}')
cv = counts.std() / counts.mean()
print('\n变异系数 CV =', round(cv, 3), '（越小越均衡；坍塌时很大）')
assert counts.sum() == T * k

> 随机初始化下尚算均匀；真实训练若不加均衡损失，CV 会逐步变大（坍塌）。

---
## ✏️ 练习 1：实现 top-k 路由

实现 `my_route(X, Wr, k)` 返回 `(topk_idx, gates)`，gates 在 top-k 上归一化。

In [ ]:
def my_route(X, Wr, k):
    # TODO: softmax -> top-k -> 归一化门控
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
ti, gg = my_route(X, Wr, 2)
assert ti.shape == (T, 2) and gg.shape == (T, 2)
assert np.allclose(gg.sum(1), 1), '门控应归一化'
assert np.all(ti[:, 0] == np.argmax(X @ Wr, axis=1)), 'top-1 应是概率最大专家'
print('✅ 练习 1 通过')

## ✏️ 练习 2：负载均衡损失

实现 `my_lb_loss(topk, probs, N, alpha)`；验证完全均匀分配时损失 ≈ α。

In [ ]:
def my_lb_loss(topk, probs, N, alpha=0.01):
    # TODO: 返回标量损失 α·N·Σ f_i·P_i
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
Nt = 6
fake_topk = np.arange(Nt).reshape(Nt, 1)
fake_probs = np.full((Nt, Nt), 1.0 / Nt)
l = my_lb_loss(fake_topk, fake_probs, Nt, 0.01)
assert abs(l - 0.01) < 1e-6, f'均匀时应≈α=0.01，得到 {l}'
print('✅ 练习 2 通过')

## ✏️ 练习 3：SwiGLU 与 ReLU-FFN 的参数对齐

实现 `swiglu_hidden_for_parity(d_model)`：返回使 SwiGLU(3 矩阵) 与 4·d_model 的 ReLU-FFN(2 矩阵) 参数量相当的隐藏维度（≈ 8/3·d_model，向 64 取整）。

In [ ]:
def swiglu_hidden_for_parity(d_model, multiple=64):
    # ReLU-FFN 参数 ≈ 2*d*(4d)。SwiGLU 参数 ≈ 3*d*h。令相等解 h，再向上取整到 multiple 的倍数。
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
hh = swiglu_hidden_for_parity(4096)
assert hh % 64 == 0
assert abs(hh - 8/3*4096) < 64, '应接近 8/3 d_model'
print('✅ 练习 3 通过 | d_model=4096 -> SwiGLU hidden =', hh)

---
### 📖 参考答案

In [ ]:
# 练习 1
def my_route(X, Wr, k):
    probs = softmax(X @ Wr, -1)
    topk = np.argsort(-probs, axis=1)[:, :k]
    gates = np.take_along_axis(probs, topk, 1)
    return topk, gates / gates.sum(1, keepdims=True)

# 练习 2
def my_lb_loss(topk, probs, N, alpha=0.01):
    T, k = topk.shape
    f = np.bincount(topk.reshape(-1), minlength=N) / (T * k)
    P = probs.mean(0)
    return alpha * N * np.sum(f * P)

# 练习 3
def swiglu_hidden_for_parity(d_model, multiple=64):
    h = (2 * d_model * 4 * d_model) / (3 * d_model)   # = 8/3 d_model
    return int(np.ceil(h / multiple) * multiple)

---
## 🧪 真实数据胶囊：真实 MoE 的稀疏度

用真实模型配置算“总参数 vs 激活参数”，体会稀疏激活的省。

In [ ]:
# 取自公开技术报告
MOE = {
    'Mixtral-8x7B':  dict(N=8,   k=2),
    'DeepSeek-V3':   dict(N=256, k=8),   # 另有共享专家，这里简化
    'Switch-Base':   dict(N=128, k=1),
}
for name, m in MOE.items():
    sparsity = m['k'] / m['N']
    print(f"{name:14s} N={m['N']:3d} k={m['k']:2d} -> 每 token 激活 {sparsity:6.1%} 的专家")
print('\nMoE 的卖点：总参数可上千亿/万亿，但每 token 只算其中很小一片。')

**🧪 胶囊练习**：实现 `active_param_fraction(N, k)` 返回激活专家占比 `k/N`。

In [ ]:
def active_param_fraction(N, k):
    # TODO
    raise NotImplementedError

In [ ]:
assert abs(active_param_fraction(8, 2) - 0.25) < 1e-9
assert abs(active_param_fraction(256, 8) - 0.03125) < 1e-9
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def active_param_fraction(N, k):
    return k / N

---
### 小结
- MoE 解耦总参数（容量）与每 token FLOPs（成本）。
- top-k 路由不可微 → 易坍塌 → 负载均衡损失 + 专家容量来救。
- SwiGLU 门控 FFN：乘性交互、等参数更强，专家 FFN 标配。

下一站：**模块 04 · RMSNorm 与稳定性**。